# ema-second-moment — ex1: Adam v-buffer EMA update v = beta2*v + (1-beta2)*g^2

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `ema-second-moment`. Running the final beacon cell reports progress against the `Optimizer: Adam EMA second moment` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA second moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-second-moment`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-second-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA second moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Adam EMA second moment `v = beta2*v + (1-beta2)*g^2` — quick refresher

Adam maintains TWO running averages per parameter: the first moment (EMA of gradients, `m`) and the second moment (EMA of squared gradients, `v`). The second moment recurrence is:

```
v_t = beta2 * v_{t-1} + (1 - beta2) * g_t^2
```

**Why squared.** `v_t` approximates the second raw moment of the gradient distribution. After bias correction, `sqrt(v_hat)` is approximately the per-coord gradient magnitude. Adam divides by this to produce its adaptive per-parameter learning rate — large-gradient coordinates get small effective lr, small-gradient coordinates get large effective lr.

**Default `beta2 = 0.999`.** That gives an effective horizon of ~1/(1-beta2) = 1000 steps. The recurrence is unbiased only in the limit; the `1/(1 - beta2^t)` bias correction at step `t` is what makes early steps usable.

### Exercise 1 — Adam v-buffer EMA update v = beta2*v + (1-beta2)*g^2

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Adam second-moment recurrence `v = beta2*v + (1-beta2)*g^2` via `buffer.copy_()` so the squared-gradient EMA buffer state is correctly mutated in place across steps.
> Keywords: adam, second-moment, ema, squared-grad
> ```

**KCs targeted:** `ema-second-moment-recurrence`, `buffer-copy_-mutates-state-in-place`

Implement `ex1_ema_v_step(v_list, grad_list, beta2)`. The second-moment update from Adam.

For each `(v, g)` pair drawn from `(v_list, grad_list)`:

1. Compute the new value: `beta2 * v + (1 - beta2) * g.pow(2)` (or equivalently `g * g`).
2. Mutate the buffer IN PLACE: `v.copy_(...)`. Don't rebind.
3. Append the new buffer value to the return list.

Inputs:
- `v_list`: list of per-param second-moment buffers (mutated).
- `grad_list`: list of per-param gradients (NOT mutated).
- `beta2`: float in `(0, 1)` — Adam default is `0.999`.

Output: list of updated `v` tensors.

The test runs three steps with KNOWN gradients and verifies that the EMA converges toward `g^2` at the rate dictated by `beta2`, AND that the buffer is in-place mutated (id and data_ptr preserved across steps).

In [ ]:
def ex1_ema_v_step(v_list: list, grad_list: list, beta2: float) -> list:
    """In-place update: v.copy_(beta2*v + (1-beta2)*g.pow(2))."""
    raise NotImplementedError()


def _test_ex1():
    # One param, zero-init buffer.
    v = t.zeros(4)
    orig_id = id(v)
    orig_ptr = v.data_ptr()

    # === Step 1: zero buffer + g => v_1 = (1 - beta2) * g^2 ===
    g1 = t.tensor([2.0, 4.0, 6.0, 8.0])
    beta2 = 0.9
    out1 = ex1_ema_v_step([v], [g1], beta2=beta2)
    expected1 = (1 - beta2) * g1.pow(2)   # 0.1 * [4, 16, 36, 64]
    assert t.allclose(out1[0], expected1), (
        f'step 1: expected {expected1}, got {out1[0]}; '
        f'check formula: beta2*v + (1-beta2)*g^2'
    )
    assert t.allclose(v, expected1), 'step 1: buffer not mutated to new value'
    assert id(v) == orig_id, 'buffer was rebound — use v.copy_(...) not v = ...'
    assert v.data_ptr() == orig_ptr, 'buffer storage reallocated'

    # === Step 2: same g; v approaches g^2 monotonically ===
    # Snapshot v BEFORE step 2 so we can compare directions after the in-place update.
    v_before_step2 = v.clone()
    out2 = ex1_ema_v_step([v], [g1], beta2=beta2)
    expected2 = beta2 * expected1 + (1 - beta2) * g1.pow(2)
    v_after_step2 = v.clone()
    assert t.allclose(v_after_step2, expected2), (
        f'step 2: expected {expected2}, got {v_after_step2}; '
        f'this fails if step 1 did NOT mutate the buffer (rebind bug)'
    )
    # v should be moving toward g^2 = [4, 16, 36, 64].
    g_sq = g1.pow(2)
    assert (v_after_step2 < g_sq).all(), 'v should still be below g^2 after only 2 steps'
    assert (v_after_step2 > v_before_step2).all(), (
        'EMA should grow toward g^2 with positive constant g; '
        'if it shrunk, your formula has the wrong sign'
    )

    # === Step 3: still moving toward g^2; spot-check direction ===
    v_before_step3 = v.clone()
    ex1_ema_v_step([v], [g1], beta2=beta2)
    v_after_step3 = v.clone()
    assert (v_after_step3 > v_before_step3).all(), 'EMA should keep growing toward g^2 with constant g'
    assert (v_after_step3 < g_sq).all(), 'EMA must not overshoot g^2'

    # === Multi-param batch ===
    v_multi = [t.zeros(2), t.zeros(3, 3)]
    g_multi = [t.tensor([1.0, -1.0]), t.ones(3, 3) * 2.0]
    ex1_ema_v_step(v_multi, g_multi, beta2=0.5)
    # 0.5 * 0 + 0.5 * [1, 1] = [0.5, 0.5]
    assert t.allclose(v_multi[0], t.tensor([0.5, 0.5])), (
        f'multi-param step: v_multi[0]={v_multi[0]}'
    )
    # 0.5 * 0 + 0.5 * 4 = 2.0 everywhere
    assert t.allclose(v_multi[1], t.full((3, 3), 2.0)), (
        f'multi-param step: v_multi[1]={v_multi[1]}'
    )

    # === Negative gradients: g^2 is positive, v stays non-negative ===
    v_neg = t.zeros(3)
    g_neg = t.tensor([-3.0, -4.0, -5.0])
    ex1_ema_v_step([v_neg], [g_neg], beta2=0.9)
    assert (v_neg >= 0).all(), (
        f'v must remain non-negative (we square g); got {v_neg}; '
        f'are you computing g.pow(2) or just g?'
    )
    expected_neg = 0.1 * t.tensor([9.0, 16.0, 25.0])
    assert t.allclose(v_neg, expected_neg), (
        f'negative-g case: expected {expected_neg}, got {v_neg}'
    )

    # === Input grad must not be mutated ===
    g_in = t.tensor([2.0, 3.0])
    g_snap = g_in.clone()
    ex1_ema_v_step([t.zeros(2)], [g_in], beta2=0.99)
    assert t.equal(g_in, g_snap), 'grad tensors must not be mutated by the EMA update'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_ema_v_step(v_list, grad_list, beta2):
    out = []
    for v, g in zip(v_list, grad_list):
        v.copy_(beta2 * v + (1 - beta2) * g.pow(2))
        out.append(v)
    return out
```

**Why `g.pow(2)` and not `g ** 2`.** Functionally identical for this case. ARENA's solution uses `g.pow(2)` to mirror the explicit pseudocode notation `g_t^2`. Both compile down to the same kernel.

**What the EMA converges to.** If `g` is constant, `v_t = (1 - beta2^t) * g^2`. As `t -> inf`, `v_t -> g^2`. That's the intuition — `v` is a low-pass filter on `g^2` with time constant `~1/(1 - beta2)`. With `beta2 = 0.999`, the effective averaging window is ~1000 recent steps.

**Bias correction is a SEPARATE drill.** The recurrence here is biased toward zero in the first few steps (because `v` starts at zero). Adam fixes this with `v_hat = v / (1 - beta2^t)`. That correction is its own atom (`bias-correction-divide`) — this drill isolates just the EMA mechanic.

**Why this matters operationally.** `v` shows up under a `sqrt` in the Adam denominator: `theta -= lr * m_hat / (sqrt(v_hat) + eps)`. Coordinates with consistently large `|g|` get large `v`, large denominator, SMALL effective lr. Coordinates with tiny `|g|` get tiny `v`, tiny denominator, LARGE effective lr. That's adaptive per-parameter learning — the whole point of Adam over plain SGD.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()